In [1]:
import mediapipe as mp
import pandas as pd
import numpy as np
import cv2
import joblib
import os

In [2]:
# === STEP 1: Extract kkeleton and save as txt ===
def extract_skeleton_txt(video_path, output_path="skeleton_lines.txt"):
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.3)
    cap = cv2.VideoCapture(video_path)
    output_txt = open(output_path, "w")

    frame_id = 0
    def get_coords(landmarks, idx):
        pt = landmarks[idx]
        return (pt.x, pt.y) if pt.visibility > 0.5 else (None, None)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose.process(frame_rgb)

        if result.pose_landmarks:
            lm = result.pose_landmarks.landmark

            l_sh, r_sh = get_coords(lm, 11), get_coords(lm, 12)
            l_hip, r_hip = get_coords(lm, 23), get_coords(lm, 24)
            l_knee, r_knee = get_coords(lm, 25), get_coords(lm, 26)
            l_ankle, r_ankle = get_coords(lm, 27), get_coords(lm, 28)

            if None not in l_sh + r_sh + l_hip + r_hip + l_knee + r_knee + l_ankle + r_ankle:
                mid_sh = [(l_sh[0] + r_sh[0]) / 2, (l_sh[1] + r_sh[1]) / 2]
                mid_hip = [(l_hip[0] + r_hip[0]) / 2, (l_hip[1] + r_hip[1]) / 2]

                line = f"{frame_id} {mid_sh[0]} {mid_sh[1]} {mid_hip[0]} {mid_hip[1]} " \
                       f"{l_hip[0]} {l_hip[1]} {r_hip[0]} {r_hip[1]} " \
                       f"{l_hip[0]} {l_hip[1]} {l_knee[0]} {l_knee[1]} {l_ankle[0]} {l_ankle[1]} " \
                       f"{r_hip[0]} {r_hip[1]} {r_knee[0]} {r_knee[1]} {r_ankle[0]} {r_ankle[1]}\n"
                output_txt.write(line)
        frame_id += 1

    cap.release()
    output_txt.close()
    pose.close()
    print(f"Saved keypoints to {output_path}")

In [3]:
# === STEP 2: Extract features from txt annotation ===
def extract_features_from_annotation(file_path):
    data = []
    with open(file_path, 'r') as f:
        lines = f.readlines()

    frames = lines[2:]  

    for line in frames:
        values = line.strip().split()
        if len(values) != 21:
            continue
        frame_data = [float(v) for v in values]
        data.append(frame_data)

    data = np.array(data)
    if data.shape[0] == 0:
        return None

    frames_num = data[:, 0]
    keypoints = data[:, 1:]

    centers_x = []
    centers_y = []
    aspect_ratios = []
    missing_counts = 0

    for kp in keypoints:
        if -1 in kp[:4]:
            missing_counts += 1
            continue
        shoulder_x, shoulder_y = kp[0], kp[1]
        hip_x, hip_y = kp[2], kp[3]
        center_x = (shoulder_x + hip_x) / 2
        center_y = (shoulder_y + hip_y) / 2
        width = abs(shoulder_x - hip_x)
        height = abs(shoulder_y - hip_y)
        aspect_ratio = width / (height + 1e-6)
        centers_x.append(center_x)
        centers_y.append(center_y)
        aspect_ratios.append(aspect_ratio)

    centers_x = np.array(centers_x)
    centers_y = np.array(centers_y)
    aspect_ratios = np.array(aspect_ratios)
    velocity_x = np.diff(centers_x)
    velocity_y = np.diff(centers_y)
    velocity_center = np.sqrt(velocity_x**2 + velocity_y**2)
    acceleration_center = np.diff(velocity_center)

    features = {
        'mean_center_x': np.mean(centers_x) if len(centers_x) > 0 else 0,
        'mean_center_y': np.mean(centers_y) if len(centers_y) > 0 else 0,
        'std_center_x': np.std(centers_x) if len(centers_x) > 0 else 0,
        'std_center_y': np.std(centers_y) if len(centers_y) > 0 else 0,
        'mean_aspect_ratio': np.mean(aspect_ratios) if len(aspect_ratios) > 0 else 0,
        'max_velocity_center': np.max(velocity_center) if len(velocity_center) > 0 else 0,
        'mean_acceleration_center': np.mean(acceleration_center) if len(acceleration_center) > 0 else 0,
        'total_horizontal_displacement': centers_x[-1] - centers_x[0] if len(centers_x) > 1 else 0,
        'total_vertical_displacement': centers_y[-1] - centers_y[0] if len(centers_y) > 1 else 0,
        'missing_data_ratio': missing_counts / len(keypoints)
    }
    return features

In [4]:
# === STEP 3: Make a prediction ===
def predict_fall(video_path, model_path, scaler_path):
    annotation_txt = "skeleton_lines.txt"
    extract_skeleton_txt(video_path, annotation_txt)
    new_features = extract_features_from_annotation(annotation_txt)

    if new_features is not None:
        X_new = np.array(list(new_features.values())).reshape(1, -1)
        scaler = joblib.load(scaler_path)
        svc_model = joblib.load(model_path)
        X_new_scaled = scaler.transform(X_new)
        prediction = svc_model.predict(X_new_scaled)
        prediction_proba = svc_model.predict_proba(X_new_scaled)
        label = 'FALL' if prediction[0] == 1 else 'NO FALL'
        print(f"\nPrediction: {label}")
        print(f"Confidence (No Fall / Fall): {prediction_proba}")
        return label
    else:
        print("No valid features extracted from the file. Cannot predict.")
        return "INVALID"

In [5]:
video_file = "test_fall.avi"  #input video
model_file = "svc_model.pkl"   #classifier
scaler_file = "scaler.pkl"     #scaler

result = predict_fall(video_file, model_file, scaler_file)

Saved keypoints to skeleton_lines.txt

Prediction: FALL
Confidence (No Fall / Fall): [[0.25702806 0.74297194]]
